### Taxi

In [0]:
from pyspark.sql import functions as F

df_taxi = spark.read.table("unified_lakehouse.bronze.taxi")

silver_taxi_df = (
    df_taxi
    .filter(
        (F.col("passenger_count") > 0) & 
        (F.col("trip_distance") > 0) & 
        (F.col("fare_amount") > 0)
    )
    .withColumn(
        "description",
        F.when(F.col("payment_type") == 1, "Credit Card")
         .when(F.col("payment_type") == 2, "Cash")
         .otherwise("Other")
    )
    .withColumn("ingest_time", F.current_timestamp())
    .dropDuplicates()
)

silver_taxi_df.write.format("delta").mode("overwrite").saveAsTable("unified_lakehouse.silver.taxi")

### Drivers

**Change**: There is not such a column as ID in drivers table so I replaced it with car_number

In [0]:
from pyspark.sql import functions as F

df_drivers = spark.read.table("unified_lakehouse.bronze.drivers")

silver_drivers_df = (
    df_drivers
    .filter(
        (F.col("experience") > 0) & 
        (F.col("rating").between(1.0, 5.0))
    )
    .withColumn("ingest_time", F.current_timestamp())
    .dropDuplicates(["car_number"])
)

silver_drivers_df.write.format("delta").mode("overwrite").saveAsTable("unified_lakehouse.silver.drivers")

### Users

In [0]:
from pyspark.sql import functions as F

df_users = spark.read.table("unified_lakehouse.bronze.users")

silver_users_df = (
    df_users
    .filter(F.col("name").isNotNull())
    .withColumn("ingest_time", F.current_timestamp())
    .withColumn("registration_date", F.current_date())
    .dropDuplicates()
)

silver_users_df.write.format("delta").mode("overwrite").saveAsTable("unified_lakehouse.silver.users")